In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(sys.prefix).parent
%cd {PROJECT_ROOT}

/home/younes/younes/Projects/Python/barid_internship


In [2]:
import polars as pl
import altair as alt

import seaborn as sns
import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
# import statsmodels.api as sm
# from statsmodels.graphics.tsaplots import plot_acf
# import calendar
# from fpppy.utils import plot_series, plot_series_stacked, plot_diagnostics
# from scipy.stats import pearsonr
# from plotly import express as px
# from pathlib import Path
# from lxml import html
# from itertools import chain

import importlib
import lib
import offline_historique
import read_data

importlib.reload(lib)
importlib.reload(offline_historique)
from offline_historique import parse_historique_from_folder  # noqa: E402

cfg = pl.Config()
cfg.set_tbl_width_chars(10000)
cfg.set_fmt_str_lengths(100)
cfg.set_tbl_cols(-1)
cfg.set_tbl_rows(20)

polars.config.Config

In [3]:
def complete_operations(
    df: pl.DataFrame,
    starting_states: list = ["depot"],
    end_states: list = ["liv","liv_ret"],
) -> pl.DataFrame:
    return (
        df
        .filter(
            pl.col("status_code").first().over("id").is_in(starting_states),
            pl.col("status_code").last().over("id").is_in(end_states),
        )
    )

In [4]:
cabs_domestic = set(read_data.read_smi_suiviexpedition_many()["CAB"])

In [5]:
fields = (
    pl.read_parquet("data/cab_dfs/fields.parquet")
    .filter(Etat="V")
    .filter(pl.col("Cab").is_in(cabs_domestic))
    .filter(pl.col("Date_depot").ge(pl.date(2023, 1, 1)))
    .cast({"Id": pl.String})
)
operations = (
    pl.read_parquet("data/cab_dfs/operations.parquet").filter(is_valid="V")
    # .sort("cab", "id", "Heure_Syst_Oper")
    .filter(pl.col("cab").is_in(cabs_domestic))
)
delivery = pl.read_parquet("data/cab_dfs/delivery.parquet").filter(
    pl.col("cab").is_in(cabs_domestic)
)
services = pl.read_parquet("data/cab_dfs/services.parquet").filter(
    pl.col("cab").is_in(cabs_domestic)
)

In [6]:
print(fields.head())

shape: (5, 21)
┌──────────────────────┬──────────┬────────────┬──────────┬────────────────┬────────┬─────────┬──────┬────────────────────┬──────────────────────────────┬─────────────┬─────────────────────────────────────────────┬────────────────────────┬────────────────────────────────────────────────────┬──────────────┬──────────────────────┬──────────────────────┬──────────┬─────────┬─────────┬────────────────────┐
│ Cab                  ┆ Id       ┆ Date_depot ┆ Type_cab ┆ Dernier_statut ┆ Regime ┆ Contrat ┆ Etat ┆ Poids_global_en_KG ┆ Centre_Agence_depot          ┆ Destination ┆ Client                                      ┆ Produit_Niveau_Service ┆ Mode_paiement                                      ┆ Taxe_DTQ_Dhs ┆ Canal_de_livraison_1 ┆ Canal_de_livraison_2 ┆ Longueur ┆ Hauteur ┆ Largeur ┆ Poids_Volumetrique │
│ ---                  ┆ ---      ┆ ---        ┆ ---      ┆ ---            ┆ ---    ┆ ---     ┆ ---  ┆ ---                ┆ ---                          ┆ ---         ┆ --- 

In [7]:
print(fields.select(pl.all().n_unique()))

shape: (1, 21)
┌────────┬────────┬────────────┬──────────┬────────────────┬────────┬─────────┬──────┬────────────────────┬─────────────────────┬─────────────┬────────┬────────────────────────┬───────────────┬──────────────┬──────────────────────┬──────────────────────┬──────────┬─────────┬─────────┬────────────────────┐
│ Cab    ┆ Id     ┆ Date_depot ┆ Type_cab ┆ Dernier_statut ┆ Regime ┆ Contrat ┆ Etat ┆ Poids_global_en_KG ┆ Centre_Agence_depot ┆ Destination ┆ Client ┆ Produit_Niveau_Service ┆ Mode_paiement ┆ Taxe_DTQ_Dhs ┆ Canal_de_livraison_1 ┆ Canal_de_livraison_2 ┆ Longueur ┆ Hauteur ┆ Largeur ┆ Poids_Volumetrique │
│ ---    ┆ ---    ┆ ---        ┆ ---      ┆ ---            ┆ ---    ┆ ---     ┆ ---  ┆ ---                ┆ ---                 ┆ ---         ┆ ---    ┆ ---                    ┆ ---           ┆ ---          ┆ ---                  ┆ ---                  ┆ ---      ┆ ---     ┆ ---     ┆ ---                │
│ u32    ┆ u32    ┆ u32        ┆ u32      ┆ u32            ┆ u32

In [8]:
df = (
    fields
    # .filter(Centre_Agence_depot="AGENCE AM MARRAKECH GUELIZ(96614)")
    .filter(
        pl.col("Cab").is_in(
            set(
                operations.sort(
                    pl.col("Heure_Syst_Oper").first().over("id"),
                    pl.col("Heure_Syst_Oper"),
                ).pipe(complete_operations, end_states=["liv"])["cab"]
            )
        )
    )
    .group_by("Centre_Agence_depot", maintain_order=True)
    .len()
    .sort("len", descending=True)
    .with_columns(
        ratio=pl.col("len") / pl.col("len").sum(),
        cum_ratio=pl.col("len").cum_sum() / pl.col("len").sum(),
    )
    .head(10)
)

chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x=alt.X("Centre_Agence_depot:N", sort=None),
        y=alt.Y("ratio"),
        tooltip=[
            "len",
            "ratio",
            "cum_ratio",
        ],
    )
    .properties(width=1100, height=400)
)
chart

alt.Chart(...)

In [9]:
print(
    operations
    # .sort(
    #     pl.col("Heure_Syst_Oper").first().over("id"), pl.col("Heure_Syst_Oper")
    # )
    .pipe(complete_operations, end_states=["liv"])
    # .group_by("id", maintain_order=True).len().mean()
    .group_by("id", maintain_order=True)
    .agg(duration=pl.col("Heure_Syst_Oper").last() - pl.col("Heure_Syst_Oper").first())
    .mean()
)

shape: (1, 2)
┌──────┬─────────────────────────┐
│ id   ┆ duration                │
│ ---  ┆ ---                     │
│ str  ┆ duration[μs]            │
╞══════╪═════════════════════════╡
│ null ┆ 4d 17h 37m 25s 600847µs │
└──────┴─────────────────────────┘


In [10]:
print(
    operations
    # .pipe(complete_operations, end_states=["aexp"])
    .filter(
        pl.col("Date_operation").lt(pl.date(2026, 1, 1)),
    )
    .group_by("id", maintain_order=True)
    .agg(
        first=pl.col("status_code").first(),
        last=pl.col("status_code").last(),
    )
    .group_by("last", maintain_order=True)
    .len()
    .sort("len", descending=True)
    .with_columns(
        cum_ratio=pl.col("len").cum_sum() / pl.col("len").sum(),
    )
)

shape: (13, 3)
┌─────────┬────────┬───────────┐
│ last    ┆ len    ┆ cum_ratio │
│ ---     ┆ ---    ┆ ---       │
│ str     ┆ u32    ┆ f64       │
╞═════════╪════════╪═══════════╡
│ liv     ┆ 218095 ┆ 0.944077  │
│ liv_ret ┆ 8173   ┆ 0.979456  │
│ aexp    ┆ 3035   ┆ 0.992594  │
│ recpt   ┆ 562    ┆ 0.995026  │
│ areturn ┆ 485    ┆ 0.997126  │
│ nrcl    ┆ 248    ┆ 0.998199  │
│ chrgctr ┆ 171    ┆ 0.998939  │
│ affg    ┆ 133    ┆ 0.999515  │
│ aff     ┆ 66     ┆ 0.999801  │
│ depot   ┆ 24     ┆ 0.999905  │
│ recg    ┆ 17     ┆ 0.999978  │
│ lev     ┆ 4      ┆ 0.999996  │
│ reexp   ┆ 1      ┆ 1.0       │
└─────────┴────────┴───────────┘


In [11]:
def map_groups(df, by_cols, fn):
    return (
        df
        # .sort(pl.col("cab").n_unique().over(by_cols))
        .group_by(by_cols, maintain_order=True)
        .map_groups(
            lambda sub_df: sub_df.pipe(fn)
            .with_columns(Destination=pl.lit(sub_df[by_cols][0]))
        )
    )

In [12]:
importlib.reload(lib)
df = (
    operations.sort(
        pl.col("Heure_Syst_Oper").first().over("id"), pl.col("Heure_Syst_Oper")
    )
    .pipe(
        complete_operations,
        # end_states=[
        #     # "liv",
        #     "aexp",
        # ],
    )
    # .join(fields["Id", "Destination"], left_on="id", right_on="Id", how="left")
    # .filter(pl.col("id").n_unique().over("Destination") >= 10000)
    .pipe(lib.span_by_id, key="id", date_col="Heure_Syst_Oper")
    .pipe(lib.survival_table, value_col="lead_days", bucket_size=1)
    # .sort(pl.col("cab").n_unique().over("Destination"))
    # .group_by("Destination", maintain_order=True)
    # .map_groups(
    #     lambda g: g.pipe(lib.span_by_id, key="id", date_col="Heure_Syst_Oper")
    #     .pipe(lib.survival_table, value_col="lead_days", bucket_size=10)
    #     .with_columns(Destination=pl.lit(g["Destination"][0]))
    # )
    # .filter(pl.col("bucket") > 0)
)

base = (
    alt.Chart(df)
    .mark_line(point=True)
    .encode(
        x=alt.X("bucket:Q", title="Lead Time Bucket (days)").scale(
            domainMin=1,
            type="log",
        ),
        tooltip=[
            alt.Tooltip("bucket:Q", title="days"),
            alt.Tooltip("cum_hazard:Q", title="cum_hazard"),
            alt.Tooltip("events:Q", title="events"),
            alt.Tooltip("survival:Q", title="survival"),
        ],
    )
    .properties(
        width=1100, height=400, title="Aggregated Delivery Lead Time Distribution"
    )
)

chart = (
    base.encode(
        y=alt.Y(
            # "pdf:Q",
            # "hazard:Q",
            "survival:Q",
        ).scale(
            type="log",
        ),
    )
    # .facet(
    #     row=alt.Row("Destination:N", title="Destination"),
    # )
)

chart

alt.Chart(...)